# Lagos Property Scraper — Nigeria Property Centre
** Web scraping (BeautifulSoup + Requests)**



## Step 1: Import libraries

In [ ]:
import re
import time
import random
from datetime import date
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

## Step 2: Settings
Which pages to scrape, the page limit, a browser-like header, and the regex patterns used to find listings, prices, property types and dates.

In [ ]:
BASE_URL = "https://nigeriapropertycentre.com"

SEARCHES = {
    "Sale": "/for-sale/lagos",
    "Rent": "/for-rent/lagos",

    
    # "Sale ": "/for-sale/houses/lagos/showtype",
    # "Rent ": "/for-rent/flats-apartments/lagos/showtype",
}

MAX_PAGES = 50                      # site limit per search
OUTPUT_FILE = "lagos_properties_raw.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# A listing link looks like: /for-sale/houses/.../lagos/lekki/3154888-newly-built-...
LISTING_LINK = re.compile(r"/for-(sale|rent)/.+/(\d{5,})-[\w-]+$")
PRICE_LINE = re.compile(r"^([₦$£€])\s?([\d,]+(?:\.\d+)?)\s*(.*)$")
TYPE_LINE = re.compile(r"^(.+?)\s+for\s+(sale|rent|short let)$", re.I)
ADDED = re.compile(r"Added\s+(today|yesterday|\d+\s+\w+\s+ago|\d{1,2}\s+\w{3}\s+\d{4})", re.I)

## Step 3: Download a page
Requests the page and retries up to 3 times if it fails. Status 200 = success.

In [ ]:
def get_page(session, url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                return resp.text
            print(f"   status {resp.status_code} (attempt {attempt})")
        except requests.RequestException as err:
            print(f"   error: {err} (attempt {attempt})")
        time.sleep(5 * attempt)
    return None

## Step 4: Find each listing card
Finds every listing link (`/for-sale/.../1234567-title`), then climbs up the HTML to the block that holds only that one listing.

In [ ]:
def listing_id(href):
    match = LISTING_LINK.search(href.split("?")[0])
    return match.group(2) if match else None


def find_cards(soup):
    """Find every listing link, then climb up the HTML tree to the
    biggest block that contains ONLY that listing - that block is the card.
    This avoids relying on CSS class names that may change."""
    cards = {}
    for a in soup.find_all("a", href=True):
        lid = listing_id(a["href"])
        if not lid or lid in cards:
            continue
        node = a
        while node.parent is not None and node.parent.name != "[document]":
            ids_in_parent = {
                listing_id(x["href"])
                for x in node.parent.find_all("a", href=True)
                if listing_id(x["href"])
            }
            if len(ids_in_parent) > 1:
                break
            node = node.parent
        cards[lid] = (node, urljoin(BASE_URL, a["href"]))
    return cards

## Step 5: Extract details from one card
Pulls out title, price, property type, location, beds, baths, toilets, parking, size and date added.

In [ ]:
def parse_card(card, lid, url, listing_type, page):
    lines = [ln for ln in card.get_text("\n", strip=True).split("\n") if ln]

    row = {
        "listing_id": lid,
        "listing_type": listing_type,
        "title": None,
        "property_type": None,
        "price_raw": None,
        "currency": None,
        "price": None,
        "price_unit": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "toilets": None,
        "parking": None,
        "size_sqm": None,
        "added": None,
        "url": url,
        "page": page,
        "scraped_on": date.today().isoformat(),
    }

    # Title (listing headings are h3/h4)
    heading = card.find(["h3", "h4"])
    if heading:
        row["title"] = heading.get_text(" ", strip=True)

    # Price and property type
    for ln in lines:
        if row["price_raw"] is None:
            m = PRICE_LINE.match(ln)
            if m:
                row["price_raw"] = ln
                row["currency"] = m.group(1)
                row["price"] = m.group(2)
                row["price_unit"] = m.group(3) or None
                continue
        if row["property_type"] is None:
            m = TYPE_LINE.match(ln)
            if m:
                row["property_type"] = m.group(1)

    # Location = last line that ends in ", Lagos" (description comes before it)
    loc_index = None
    for i, ln in enumerate(lines):
        if re.search(r",?\s*Lagos\b", ln) and not ln.lower().startswith(("₦", "$")):
            if re.search(r"Lagos(\s+Added.*)?$", ln):
                loc_index = i
    if loc_index is not None:
        row["location"] = re.sub(r"\s+Added\s+.*$", "", lines[loc_index]).strip()

    # "Added today / yesterday / ..." (only shown on recent listings)
    m = ADDED.search(" ".join(lines))
    if m:
        row["added"] = m.group(1)

    # Features (beds, baths, toilets, parking, sqm) appear AFTER the location
    feature_text = " ".join(lines[loc_index + 1:] if loc_index is not None else lines)
    patterns = {
        "bedrooms": r"(\d+)\s*beds?\b",
        "bathrooms": r"(\d+)\s*baths?\b",
        "toilets": r"(\d+)\s*toilets?\b",
        "parking": r"(\d+)\s*parking\b",
        "size_sqm": r"([\d,]+)\s*sqm\b",
    }
    for field, pat in patterns.items():
        m = re.search(pat, feature_text, re.I)
        if m:
            row[field] = m.group(1)

    return row

## Step 6: Test on page 1 only
Check this works before running the full scrape. You should see about 21 listings.

In [ ]:
session = requests.Session()
html = get_page(session, BASE_URL + "/for-sale/lagos")
soup = BeautifulSoup(html, "html.parser")
cards = find_cards(soup)
print(f"Listings found on page 1: {len(cards)}")

test_rows = [parse_card(card, lid, link, "Sale", 1) for lid, (card, link) in cards.items()]
pd.DataFrame(test_rows).head(10)

## Step 7: Run the full scrape
About 5–8 minutes. Progress is saved to the CSV after every page.

In [ ]:
def main():
    session = requests.Session()
    all_rows, seen = [], set()

    for listing_type, path in SEARCHES.items():
        listing_type = listing_type.strip()
        print(f"\n=== Scraping {listing_type}: {path} ===")

        for page in range(1, MAX_PAGES + 1):
            url = BASE_URL + path + ("" if page == 1 else f"?page={page}")
            html = get_page(session, url)
            if html is None:
                print(f"Page {page}: failed, stopping this search.")
                break

            soup = BeautifulSoup(html, "html.parser")
            cards = find_cards(soup)

            if not cards:
                if page == 1:
                    with open("debug_page.html", "w", encoding="utf-8") as f:
                        f.write(html)
                    print("No listings found on page 1 - saved debug_page.html. "
                          "Send that file to Claude to adjust the parser.")
                print(f"Page {page}: no listings, stopping this search.")
                break

            new = 0
            for lid, (card, link) in cards.items():
                if lid in seen:
                    continue
                seen.add(lid)
                all_rows.append(parse_card(card, lid, link, listing_type, page))
                new += 1

            print(f"Page {page}: {new} new listings (total {len(all_rows)})")

            # Save progress after every page so nothing is lost if it stops
            pd.DataFrame(all_rows).to_csv(OUTPUT_FILE, index=False)

            time.sleep(random.uniform(2, 4))   # be polite to the server

    df = pd.DataFrame(all_rows)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nDone. {len(df)} listings saved to {OUTPUT_FILE}")
    print(df.head())
    print("\nMissing values per column:")
    print(df.isna().sum())


main()

## Step 8: Load and preview the raw data

In [ ]:
df = pd.read_csv(OUTPUT_FILE)
print(df.shape)
df.head()